## Ingestion del archivo "language.csv"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

###Paso 1 - Leer el archivo CSV usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_schema = StructType([
    StructField("languageId", IntegerType(), False),
    StructField("languageCode", StringType(), True),
    StructField("languageName", StringType(), True)
    
    
])

In [0]:
language_df = spark.read. \
    option("header", True) \
    .csv(f"{bronze_folder_path}/{v_file_date}/language.csv", schema=language_schema)

### Paso 2 - Seleccionar sólo las columnas "requeridas"

In [0]:
selected_language_df = language_df.select(language_df.languageId, language_df.languageName)

### Paso 3 - Cambiar el nombre de las columnas según lo "requerido"

In [0]:
renamed_language_df = selected_language_df.withColumnsRenamed({"languageId" : "language_id", "languageName" : "language_name"})

### Paso 4 - Agregar la columna "ingestion_date" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
final_language_df = add_ingestion_date(renamed_language_df)\
                    .withColumn("environment", lit(v_environment))\
                    .withColumn("file_date", lit(v_file_date))

### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#final_language_df.write.parquet(f"{silver_folder_path}/language", mode="overwrite")

In [0]:
final_language_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages")

In [0]:
display(final_language_df)

language_id,language_name,ingestion_date,environment,file_date
24574,English,2026-09-08T01:14:32.602763Z,production,2024-12-16
24575,svenska,2026-09-08T01:14:32.602763Z,production,2024-12-16
24576,Deutsch,2026-09-08T01:14:32.602763Z,production,2024-12-16
24577,unknown,2026-09-08T01:14:32.602763Z,production,2024-12-16
24578,Nihongo,2026-09-08T01:14:32.602763Z,production,2024-12-16
24579,Français,2026-09-08T01:14:32.602763Z,production,2024-12-16
24580,Español,2026-09-08T01:14:32.602763Z,production,2024-12-16
24581,al-?arabiyyah,2026-09-08T01:14:32.602763Z,production,2024-12-16
24582,Latin,2026-09-08T01:14:32.602763Z,production,2024-12-16
24583,Khémôrôphéasa,2026-09-08T01:14:32.602763Z,production,2024-12-16


In [0]:
dbutils.notebook.exit("Exitoso")